<a href="https://colab.research.google.com/github/12thFulminata/llm-foundations/blob/main/01-api-basics/01_gemini_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import json

drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/Colab Notebooks/res-gestae/suetonius.jsonl"

with open(DATA, encoding="utf-8") as f:
    corpus = [json.loads(line) for line in f]

print(len(corpus), "chapters loaded")
print(corpus[0]["life"], corpus[0]["chapter"], corpus[0]["text"][:100])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
541 chapters loaded
Julius 1 Julius Caesar, the Divine, lost his father when he was in the sixteenth year of his age; and the yea


In [2]:
!pip install -q google-genai

from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

for m in client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models

In [3]:
import time
from google import genai
from google.genai import types, errors
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

def ask(model, prompt, retries=4, **kwargs):
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=model, contents=prompt, **kwargs)
        except errors.ServerError as e:
            wait = 5 * 2 ** attempt
            print(f"  {model}: server error {e.code}, retrying in {wait}s...")
            time.sleep(wait)
        except errors.ClientError as e:
            if e.code == 429:
                wait = 15 * 2 ** attempt
                print(f"  {model}: rate limited, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"{model} failed after {retries} attempts")

question = "In two sentences: why did Julius Caesar cross the Rubicon?"

In [4]:


for model in ["gemini-3.5-flash-lite", "gemma-4-26b-a4b-it"]:
    response = ask(model, question)
    u = response.usage_metadata
    print(f"=== {model} ===")
    print(response.text.strip())
    print(f"tokens: prompt={u.prompt_token_count}, answer={u.candidates_token_count}, "
          f"thinking={u.thoughts_token_count}, total={u.total_token_count}\n")

=== gemini-3.5-flash-lite ===
Julius Caesar crossed the Rubicon in 49 BCE to defy the Roman Senate, which had ordered him to disband his army and return to Rome as a private citizen. By bringing his armed legions across the river, he committed an act of treason that directly triggered a civil war and ultimately led to his rise as dictator.
tokens: prompt=14, answer=65, thinking=None, total=79

=== gemma-4-26b-a4b-it ===
Julius Caesar crossed the Rubicon to avoid being stripped of his military command and facing legal prosecution by his political rivals in the Senate. By leading his army into Italy, he defied the government's orders and initiated a civil war to secure his personal power and political survival.
tokens: prompt=14, answer=55, thinking=666, total=735



In [5]:
for text in ["In two sentences: why did Julius Caesar cross the Rubicon?",
             "Rubicon", " Rubicon", "Julius Caesar", "Suetonius", "Divi filius",
             "LXXXIX", "?"]:
    n = client.models.count_tokens(model="gemini-3.5-flash-lite", contents=text).total_tokens
    print(f"{n:3} tokens  {text!r}")

 14 tokens  'In two sentences: why did Julius Caesar cross the Rubicon?'
  3 tokens  'Rubicon'
  3 tokens  ' Rubicon'
  4 tokens  'Julius Caesar'
  4 tokens  'Suetonius'
  5 tokens  'Divi filius'
  4 tokens  'LXXXIX'
  2 tokens  '?'


In [6]:
from google.genai import types

def ask(model, prompt, retries=4, **kwargs):
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=model, contents=prompt, **kwargs)
        except errors.ServerError as e:
            wait = 5 * 2 ** attempt
            print(f"  {model}: server error {e.code}, retrying in {wait}s...")
            time.sleep(wait)
        except errors.ClientError as e:
            if e.code == 429:
                wait = 15 * 2 ** attempt
                print(f"  {model}: rate limited, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"{model} failed after {retries} attempts")

cfg = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(include_thoughts=True)
)
r = ask("gemma-4-26b-a4b-it", question, config=cfg)

for part in r.candidates[0].content.parts:
    label = "THOUGHT" if part.thought else "ANSWER"
    print(f"--- {label} ---\n{part.text}\n")

  gemma-4-26b-a4b-it: server error 500, retrying in 5s...
--- THOUGHT ---

*   Subject: Julius Caesar crossing the Rubicon.
    *   Constraint 1: Exactly two sentences.
    *   Constraint 2: Explain *why* he did it.

    *   Caesar was a general/politician in Rome.
    *   He had completed his Gallic Wars.
    *   The Senate (led by Pompey and the Optimates) ordered him to disband his army and return to Rome to face trial/political challenges.
    *   Crossing the Rubicon meant declaring war on the Roman Republic/Senate to protect his own power, status, and safety.

    *   *Sentence 1:* Julius Caesar crossed the Rubicon to avoid being stripped of his command and facing political prosecution by his enemies in the Senate.
    *   *Sentence 2:* By marching his army into Italy, he effectively declared civil war to seize absolute power and defend his political interests.

    *   *Draft 1:* Julius Caesar crossed the Rubicon to evade political prosecution and the loss of his command ordered